# 03 - Creacion del target `caida_critica`

Objetivo: crear formalmente la variable objetivo del proyecto de clasificacion binaria.

Contexto: el EDA recomendo construir el target usando una variable de `produccion_principal` por pozo: `prod_pet` para pozos petroliferos y `prod_gas` para pozos gasiferos. El umbral inicial recomendado para explorar `caida_critica` es una caida futura mayor o igual al 30%.

Reglas de esta etapa:
- No se entrenan modelos.
- No se crean lags, medias moviles ni features avanzadas.
- Las variables que miran el mes siguiente se usan solo para construir el target.
- No se usan variables futuras como features.
- Se dejan fuera de `df_target` los registros donde no puede calcularse el target.

## 1. Importacion de librerias y configuracion

Se importan librerias basicas para procesamiento tabular y manejo de paths. No se importan librerias de modelado porque en este notebook no se entrenan modelos.

In [64]:
from pathlib import Path
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

TARGET_THRESHOLD_PCT = 30

## 2. Carga del dataset base

Se carga la base procesada generada en el notebook de calidad de datos. Esta base mantiene la unidad de analisis pozo-mes.

In [65]:
DATA_PATH = Path("../data/processed/produccion_no_convencional_base.csv")

df_base = pd.read_csv(DATA_PATH, low_memory=False, parse_dates=["fecha_data", "fechaingreso"])
df_base["fecha_data"] = pd.to_datetime(df_base["fecha_data"], errors="coerce")

df_fe = df_base.copy()

print(f"Filas: {df_fe.shape[0]:,}")
print(f"Columnas: {df_fe.shape[1]:,}")
df_fe.head()

Filas: 405,996
Columnas: 40


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,iny_otro,tef,vida_util,tipoextraccion,tipoestado,tipopozo,observaciones,fechaingreso,rectificado,habilitado,idusuario,empresa,sigla,formprod,profundidad,formacion,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data
0,YSUR,2016,1,135204,0.00,59.94,28.35,0.00,0.00,0.00,0.00,30.81,NaN,Plunger Lift,Extracción Efectiva,Gasífero,NaN,2016-02-17 10:50:46.929347,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.ACO-13(d),PREC,"3,500.00",precuyo,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,NEUQUINA,Neuquén,-69.79,-38.97,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2016-01-31
1,YSUR,2018,1,155584,80.81,786.90,0.00,0.00,0.00,0.00,0.00,31.00,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2018-02-10 08:37:14.717426,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,YSR.RN.EFO-262(d),LAJA,"3,847.00",lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.80,-39.03,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2018-01-31
2,YSUR,2015,1,135203,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,Sin Sistema de Extracción,Parado Transitoriamente,Gasífero,NaN,2015-02-26 13:35:35.533458,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.ACO-12(d),PREC,"3,617.00",precuyo,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,NEUQUINA,Neuquén,-69.81,-38.96,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
3,YSUR,2017,1,155582,58.28,608.01,13.63,0.00,0.00,0.00,0.00,31.00,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2017-02-16 13:45:37.233373,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,YSR.RN.EFO-245(d),LAJA,"3,805.00",lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.85,-39.01,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2017-01-31
4,YSUR,2016,1,136137,0.00,432.17,45.52,0.00,0.00,0.00,0.00,31.00,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2016-02-17 10:50:46.929347,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.Gu-1199(d),PREC,"3,375.00",precuyo,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO,NEUQUINA,Neuquén,-69.22,-38.87,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2016-01-31


## 3. Revision de valores negativos

Antes de crear el target, se identifican valores negativos en variables productivas. No se eliminan automaticamente. Para construir el target, los registros donde la produccion principal actual sea negativa quedan como no validos porque no permiten una interpretacion productiva confiable.

In [66]:
production_cols = ["prod_pet", "prod_gas", "prod_agua"]

negative_summary = pd.DataFrame(
    {
        "variable": production_cols,
        "casos_negativos": [(df_fe[col] < 0).sum() for col in production_cols],
        "valor_minimo": [df_fe[col].min() for col in production_cols],
    }
)

negative_summary

,variable,casos_negativos,valor_minimo
0,prod_pet,1,-0.00
1,prod_gas,2,-12.27
2,prod_agua,0,0.00


In [67]:
negative_mask = (df_fe[production_cols] < 0).any(axis=1)
negative_display_cols = [
    "idpozo", "anio", "mes", "fecha_data", "empresa", "provincia", "cuenca",
    "tipopozo", "tipoestado", "tipoextraccion", "prod_pet", "prod_gas", "prod_agua",
]
negative_records = df_fe.loc[negative_mask, negative_display_cols]
negative_records

,idpozo,anio,mes,fecha_data,empresa,provincia,cuenca,tipopozo,tipoestado,tipoextraccion,prod_pet,prod_gas,prod_agua
289972,153227,2020,5,2020-05-31,PLUSPETROL S.A.,Neuquén,NEUQUINA,Gasífero,Extracción Efectiva,Surgencia Natural,1.02,-7.52,0.00
290026,153228,2020,5,2020-05-31,PLUSPETROL S.A.,Neuquén,NEUQUINA,Gasífero,Extracción Efectiva,Surgencia Natural,1.67,-12.27,0.00
312211,73804,2008,12,2008-12-31,PETROBRAS ARGENTINA S.A.,Santa Cruz,AUSTRAL,Gasífero,En Estudio,Sin Sistema de Extracción,-0.00,0.00,0.00


## 4. Creacion de `produccion_principal`

Se define una variable productiva principal segun `tipopozo`:
- si el pozo es petrolifero, se usa `prod_pet`;
- si el pozo es gasifero, se usa `prod_gas`;
- para otros tipos de pozo, se deja `NaN`.

Se normaliza el texto de `tipopozo` para reconocer variantes con y sin tilde: `Petrolífero`, `Petrolifero`, `Gasífero`, `Gasifero`.

In [68]:
def normalize_text_for_matching(value: object) -> str:
    if pd.isna(value):
        return ""
    normalized = unicodedata.normalize("NFKD", str(value))
    ascii_text = normalized.encode("ascii", "ignore").decode("ascii")
    return ascii_text.strip().lower()


tipopozo_normalizado = df_fe["tipopozo"].map(normalize_text_for_matching)

petrolifero_mask = tipopozo_normalizado.eq("petrolifero")
gasifero_mask = tipopozo_normalizado.eq("gasifero")

df_fe["produccion_principal"] = np.nan
df_fe["criterio_produccion_principal"] = "fuera_definicion"

df_fe.loc[petrolifero_mask, "produccion_principal"] = df_fe.loc[petrolifero_mask, "prod_pet"]
df_fe.loc[petrolifero_mask, "criterio_produccion_principal"] = "prod_pet_por_pozo_petrolifero"

df_fe.loc[gasifero_mask, "produccion_principal"] = df_fe.loc[gasifero_mask, "prod_gas"]
df_fe.loc[gasifero_mask, "criterio_produccion_principal"] = "prod_gas_por_pozo_gasifero"

df_fe[["tipopozo", "prod_pet", "prod_gas", "produccion_principal", "criterio_produccion_principal"]].head()

,tipopozo,prod_pet,prod_gas,produccion_principal,criterio_produccion_principal
0,Gasífero,0.00,59.94,59.94,prod_gas_por_pozo_gasifero
1,Gasífero,80.81,786.90,786.90,prod_gas_por_pozo_gasifero
2,Gasífero,0.00,0.00,0.00,prod_gas_por_pozo_gasifero
3,Gasífero,58.28,608.01,608.01,prod_gas_por_pozo_gasifero
4,Gasífero,0.00,432.17,432.17,prod_gas_por_pozo_gasifero


## 5. Cobertura de la definicion de produccion principal

Se reporta cuanta informacion queda cubierta por la definicion y cuantos registros quedan fuera por no ser pozos petroliferos ni gasiferos.

In [69]:
registros_petroliferos = int(petrolifero_mask.sum())
registros_gasiferos = int(gasifero_mask.sum())
registros_fuera_definicion = int((~(petrolifero_mask | gasifero_mask)).sum())
pct_principal_nula = df_fe["produccion_principal"].isna().mean() * 100
registros_principal_no_positiva = int((df_fe["produccion_principal"] <= 0).sum())
registros_principal_negativa = int((df_fe["produccion_principal"] < 0).sum())

principal_definition_summary = pd.DataFrame(
    {
        "metrica": [
            "registros_petroliferos",
            "registros_gasiferos",
            "registros_fuera_definicion",
            "pct_produccion_principal_nula",
            "registros_produccion_principal_menor_o_igual_cero",
            "registros_produccion_principal_negativa",
        ],
        "valor": [
            registros_petroliferos,
            registros_gasiferos,
            registros_fuera_definicion,
            pct_principal_nula,
            registros_principal_no_positiva,
            registros_principal_negativa,
        ],
    }
)

principal_definition_summary

,metrica,valor
0,registros_petroliferos,"156,071.00"
1,registros_gasiferos,"221,257.00"
2,registros_fuera_definicion,"28,668.00"
3,pct_produccion_principal_nula,7.06
4,registros_produccion_principal_menor_o_igual_cero,"53,434.00"
5,registros_produccion_principal_negativa,2.00


In [70]:
principal_by_tipopozo = (
    df_fe.groupby(["tipopozo", "criterio_produccion_principal"], dropna=False)
    .agg(
        registros=("idpozo", "size"),
        pozos_unicos=("idpozo", "nunique"),
    )
    .reset_index()
    .sort_values("registros", ascending=False)
)

principal_by_tipopozo

,tipopozo,criterio_produccion_principal,registros,pozos_unicos
0,Gasífero,prod_gas_por_pozo_gasifero,221257,2448
4,Petrolífero,prod_pet_por_pozo_petrolifero,156071,2565
3,Otro tipo,fuera_definicion,27320,868
5,Sumidero,fuera_definicion,644,24
6,NaN,fuera_definicion,605,192
1,Inyección de Agua,fuera_definicion,56,4
2,Inyección de Gas,fuera_definicion,43,7


## 6. Produccion del mes siguiente y variacion futura

Se ordena el dataset por `idpozo` y `fecha_data`. Luego se calcula `produccion_principal_next` con `shift(-1)` dentro de cada pozo.

Estas variables miran el mes siguiente. Por lo tanto, solo se usan para construir `caida_critica` y no deben usarse como features en modelado.

In [71]:
df_fe = df_fe.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)

df_fe["produccion_principal_next"] = df_fe.groupby("idpozo")["produccion_principal"].shift(-1)
df_fe["var_futura_principal_pct"] = np.nan

variation_valid_mask = (
    (df_fe["produccion_principal"] > 0)
    & df_fe["produccion_principal_next"].notna()
)

df_fe.loc[variation_valid_mask, "var_futura_principal_pct"] = (
    (
        df_fe.loc[variation_valid_mask, "produccion_principal_next"]
        - df_fe.loc[variation_valid_mask, "produccion_principal"]
    )
    / df_fe.loc[variation_valid_mask, "produccion_principal"]
    * 100
)

df_fe[["idpozo", "fecha_data", "produccion_principal", "produccion_principal_next", "var_futura_principal_pct"]].head(10)

,idpozo,fecha_data,produccion_principal,produccion_principal_next,var_futura_principal_pct
0,3640,2006-02-28,0.00,30.81,NaN
1,3640,2006-03-31,30.81,76.61,148.67
2,3640,2006-04-30,76.61,63.27,-17.41
3,3640,2006-05-31,63.27,55.04,-13.01
4,3640,2006-06-30,55.04,58.86,6.94
5,3640,2006-07-31,58.86,33.24,-43.53
6,3640,2006-08-31,33.24,37.29,12.17
7,3640,2006-09-30,37.29,64.17,72.08
8,3640,2006-10-31,64.17,62.86,-2.04
9,3640,2006-11-30,62.86,66.64,6.02


## 7. Creacion de `caida_critica`

Se define `caida_critica` con el umbral exploratorio recomendado en EDA:
- `caida_critica = 1` si `var_futura_principal_pct <= -30`;
- `caida_critica = 0` si `var_futura_principal_pct > -30`.

Los registros donde no se puede calcular `var_futura_principal_pct` quedan como no modelables en esta version.

In [72]:
df_fe["caida_critica"] = pd.Series(pd.NA, index=df_fe.index, dtype="Int64")

target_calculable_mask = df_fe["var_futura_principal_pct"].notna()

df_fe.loc[target_calculable_mask, "caida_critica"] = (
    df_fe.loc[target_calculable_mask, "var_futura_principal_pct"] <= -TARGET_THRESHOLD_PCT
).astype(int)

df_fe.loc[target_calculable_mask, [
    "idpozo", "fecha_data", "produccion_principal", "produccion_principal_next",
    "var_futura_principal_pct", "caida_critica"
]].head(10)

,idpozo,fecha_data,produccion_principal,produccion_principal_next,var_futura_principal_pct,caida_critica
1,3640,2006-03-31,30.81,76.61,148.67,0
2,3640,2006-04-30,76.61,63.27,-17.41,0
3,3640,2006-05-31,63.27,55.04,-13.01,0
4,3640,2006-06-30,55.04,58.86,6.94,0
5,3640,2006-07-31,58.86,33.24,-43.53,1
6,3640,2006-08-31,33.24,37.29,12.17,0
7,3640,2006-09-30,37.29,64.17,72.08,0
8,3640,2006-10-31,64.17,62.86,-2.04,0
9,3640,2006-11-30,62.86,66.64,6.02,0
10,3640,2006-12-31,66.64,51.73,-22.38,0


## 8. Registros modelables y no modelables

`df_target` contiene solo registros donde el target pudo calcularse. Para reducir riesgo de data leakage en etapas posteriores, no se guardan en `df_target` las columnas que miran el mes siguiente: `produccion_principal_next` y `var_futura_principal_pct`.

In [73]:
not_modelable_summary = pd.DataFrame(
    {
        "motivo": [
            "produccion_principal_nula_fuera_definicion",
            "produccion_principal_menor_o_igual_cero",
            "sin_mes_siguiente_observable",
            "total_sin_target_calculable",
        ],
        "registros": [
            int(df_fe["produccion_principal"].isna().sum()),
            int((df_fe["produccion_principal"].notna() & (df_fe["produccion_principal"] <= 0)).sum()),
            int(((df_fe["produccion_principal"] > 0) & df_fe["produccion_principal_next"].isna()).sum()),
            int((~target_calculable_mask).sum()),
        ],
    }
)

not_modelable_summary

,motivo,registros
0,produccion_principal_nula_fuera_definicion,28668
1,produccion_principal_menor_o_igual_cero,53434
2,sin_mes_siguiente_observable,4173
3,total_sin_target_calculable,86275


In [74]:
future_target_columns = ["produccion_principal_next", "var_futura_principal_pct"]

df_target = df_fe.loc[target_calculable_mask].copy()
df_target["caida_critica"] = df_target["caida_critica"].astype(int)
df_target = df_target.drop(columns=future_target_columns)

print(f"Dimensiones df_fe: {df_fe.shape[0]:,} filas x {df_fe.shape[1]:,} columnas")
print(f"Dimensiones df_target: {df_target.shape[0]:,} filas x {df_target.shape[1]:,} columnas")

Dimensiones df_fe: 405,996 filas x 45 columnas
Dimensiones df_target: 319,721 filas x 43 columnas


## 9. Resumen final del target

Se reportan dimensiones, registros eliminados por falta de target, distribucion de clases, rango temporal y cantidad de pozos unicos en la base modelable.

In [75]:
target_distribution = pd.DataFrame(
    {
        "cantidad": df_target["caida_critica"].value_counts().sort_index(),
        "porcentaje": df_target["caida_critica"].value_counts(normalize=True).sort_index() * 100,
    }
)
target_distribution.index.name = "caida_critica"

target_distribution

,cantidad,porcentaje
caida_critica,,
0,281911,88.17
1,37810,11.83


In [76]:
target_final_summary = pd.DataFrame(
    {
        "metrica": [
            "filas_df_fe",
            "columnas_df_fe",
            "filas_df_target",
            "columnas_df_target",
            "registros_eliminados_por_falta_de_target",
            "fecha_min_df_target",
            "fecha_max_df_target",
            "pozos_unicos_df_target",
        ],
        "valor": [
            df_fe.shape[0],
            df_fe.shape[1],
            df_target.shape[0],
            df_target.shape[1],
            df_fe.shape[0] - df_target.shape[0],
            df_target["fecha_data"].min(),
            df_target["fecha_data"].max(),
            df_target["idpozo"].nunique(),
        ],
    }
)

target_final_summary

,metrica,valor
0,filas_df_fe,405996
1,columnas_df_fe,45
2,filas_df_target,319721
3,columnas_df_target,43
4,registros_eliminados_por_falta_de_target,86275
5,fecha_min_df_target,2006-01-31 00:00:00
6,fecha_max_df_target,2026-03-31 00:00:00
7,pozos_unicos_df_target,4681


## 10. Guardado de `df_target`

Se guarda la base modelable con el target `caida_critica`. Las columnas que miran el futuro se usaron para construir el target, pero no se guardan en `df_target` para evitar que luego sean usadas como features por error.

In [77]:
OUTPUT_PATH = Path("../data/processed/df_target_caida_critica.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_target.to_csv(OUTPUT_PATH, index=False)

pd.DataFrame(
    {
        "archivo_guardado": [str(OUTPUT_PATH)],
        "filas": [df_target.shape[0]],
        "columnas": [df_target.shape[1]],
    }
)

,archivo_guardado,filas,columnas
0,..\data\processed\df_target_caida_critica.csv,319721,43


## Conclusiones sobre creacion del target

- **Construccion de `produccion_principal`:** se uso `prod_pet` para registros de pozos petroliferos y `prod_gas` para registros de pozos gasiferos, segun `tipopozo`. Las variantes con y sin tilde fueron contempladas mediante normalizacion de texto. Otros tipos de pozo quedaron con `produccion_principal = NaN`.
- **Construccion de `caida_critica`:** se calculo la produccion principal del mes siguiente por `idpozo`, ordenando por `fecha_data`. Luego se calculo la variacion futura porcentual y se definio `caida_critica = 1` cuando la caida fue mayor o igual al 30%, es decir, `var_futura_principal_pct <= -30`. Si la variacion fue mayor a `-30`, se definio `caida_critica = 0`.
- **Registros fuera de `df_target`:** quedaron fuera los registros sin produccion principal definida, los registros con produccion principal actual menor o igual a cero y los registros sin mes siguiente observable para el mismo pozo. Tambien quedan fuera, para el calculo del target, los registros donde la produccion principal actual es negativa.
- **Balance final del target:** la clase `0` representa aproximadamente el 88,17% de los registros modelables y la clase `1` aproximadamente el 11,83%. Hay desbalance, pero no es extremo para una primera version de clasificacion binaria.
- **Variables futuras que no deben usarse como features:** `produccion_principal_next` y `var_futura_principal_pct` miran el mes siguiente. Se usaron solo para construir `caida_critica` y no se guardaron en `df_target`.
- **Cuidados pendientes para feature engineering:** las proximas features deben construirse usando solo informacion disponible hasta el mes actual. Lags, medias moviles y tendencias deberan calcularse por `idpozo`, respetando el orden temporal y evitando cualquier agregacion que incluya meses futuros.
- **Prevencion de data leakage:** el target mira el mes siguiente, pero las features no pueden mirar el mes siguiente. En modelado tambien sera necesario usar un split temporal para simular el escenario real de prediccion.

## Validacion de continuidad mensual

El target `caida_critica` fue creado con `shift(-1)` por `idpozo`. Esta validacion revisa si el siguiente registro observado para cada pozo corresponde realmente al mes calendario siguiente.

No se modifica la definicion actual del target en esta seccion. El objetivo es diagnosticar continuidad temporal y documentar una decision metodologica pendiente antes de avanzar con feature engineering y modelado.

### Calculo del siguiente registro observado

Se ordena por `idpozo` y `fecha_data`, y se crea `fecha_siguiente_observada` con `shift(-1)` dentro de cada pozo.

In [78]:
df_continuidad = df_fe.copy()
df_continuidad["fecha_data"] = pd.to_datetime(df_continuidad["fecha_data"], errors="coerce")
df_continuidad = df_continuidad.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)

df_continuidad["fecha_siguiente_observada"] = df_continuidad.groupby("idpozo")["fecha_data"].shift(-1)

mes_actual_idx = df_continuidad["fecha_data"].dt.year * 12 + df_continuidad["fecha_data"].dt.month
mes_siguiente_idx = (
    df_continuidad["fecha_siguiente_observada"].dt.year * 12
    + df_continuidad["fecha_siguiente_observada"].dt.month
)

df_continuidad["gap_meses_hasta_siguiente"] = mes_siguiente_idx - mes_actual_idx

df_continuidad[["idpozo", "fecha_data", "fecha_siguiente_observada", "gap_meses_hasta_siguiente"]].head(10)

,idpozo,fecha_data,fecha_siguiente_observada,gap_meses_hasta_siguiente
0,3640,2006-02-28,2006-03-31,1.00
1,3640,2006-03-31,2006-04-30,1.00
2,3640,2006-04-30,2006-05-31,1.00
3,3640,2006-05-31,2006-06-30,1.00
4,3640,2006-06-30,2006-07-31,1.00
5,3640,2006-07-31,2006-08-31,1.00
6,3640,2006-08-31,2006-09-30,1.00
7,3640,2006-09-30,2006-10-31,1.00
8,3640,2006-10-31,2006-11-30,1.00
9,3640,2006-11-30,2006-12-31,1.00


### Resumen de continuidad mensual

Se calculan pares con siguiente registro observado. Los ultimos registros de cada pozo quedan sin `fecha_siguiente_observada` y no forman parte del denominador de continuidad.

In [79]:
gap_valid_mask = df_continuidad["gap_meses_hasta_siguiente"].notna()
total_pares_con_siguiente = int(gap_valid_mask.sum())

pares_gap_1 = int((df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"] == 1).sum())
pares_gap_mayor_1 = int((df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"] > 1).sum())
pares_gap_otro = int((df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"] <= 0).sum())
registros_sin_siguiente = int(df_continuidad["fecha_siguiente_observada"].isna().sum())

continuidad_summary = pd.DataFrame(
    {
        "categoria": [
            "gap_1_mes",
            "gap_mayor_a_1_mes",
            "gap_menor_o_igual_a_0",
            "sin_siguiente_observado",
        ],
        "cantidad": [
            pares_gap_1,
            pares_gap_mayor_1,
            pares_gap_otro,
            registros_sin_siguiente,
        ],
        "porcentaje_sobre_pares_con_siguiente": [
            pares_gap_1 / total_pares_con_siguiente * 100,
            pares_gap_mayor_1 / total_pares_con_siguiente * 100,
            pares_gap_otro / total_pares_con_siguiente * 100,
            np.nan,
        ],
    }
)

continuidad_summary

,categoria,cantidad,porcentaje_sobre_pares_con_siguiente
0,gap_1_mes,400236,99.79
1,gap_mayor_a_1_mes,831,0.21
2,gap_menor_o_igual_a_0,0,0.00
3,sin_siguiente_observado,4929,NaN


In [80]:
gap_distribution = (
    df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"]
    .value_counts()
    .sort_index()
    .rename_axis("gap_meses_hasta_siguiente")
    .reset_index(name="cantidad")
)

gap_distribution.head(20)

,gap_meses_hasta_siguiente,cantidad
0,1.00,400236
1,2.00,610
2,3.00,44
3,4.00,13
4,5.00,7
5,6.00,5
6,7.00,8
7,8.00,5
8,9.00,3
9,10.00,1


### Cruces de gaps mayores a 1 mes

Se analizan los casos donde el siguiente registro observado no corresponde al mes calendario siguiente. Esto ayuda a detectar si los gaps se concentran en ciertos periodos, estados, tipos de pozo o empresas.

In [81]:
gaps_mayores_1 = df_continuidad.loc[df_continuidad["gap_meses_hasta_siguiente"] > 1].copy()
gaps_mayores_1["anio_gap"] = gaps_mayores_1["fecha_data"].dt.year

gaps_por_anio = (
    gaps_mayores_1.groupby("anio_gap")
    .size()
    .rename("cantidad_gaps_mayores_1")
    .reset_index()
    .sort_values("cantidad_gaps_mayores_1", ascending=False)
)

gaps_por_anio.head(20)

,anio_gap,cantidad_gaps_mayores_1
7,2014,218
15,2024,164
10,2018,156
1,2008,136
11,2019,31
16,2025,30
8,2015,18
14,2023,18
5,2012,16
6,2013,13


In [82]:
gaps_por_tipoestado = (
    gaps_mayores_1["tipoestado"]
    .value_counts(dropna=False)
    .rename_axis("tipoestado")
    .reset_index(name="cantidad_gaps_mayores_1")
)

gaps_por_tipoestado.head(15)

,tipoestado,cantidad_gaps_mayores_1
0,Extracción Efectiva,295
1,Abandonado,226
2,En Estudio,140
3,Parado Transitoriamente,73
4,Otras Situación Inactivo,41
5,En Reserva de Gas,38
6,NaN,7
7,A Abandonar,6
8,Abandono Temporario,2
9,En Reserva para Recup. Sec./Asist.,1


In [83]:
gaps_por_tipopozo = (
    gaps_mayores_1["tipopozo"]
    .value_counts(dropna=False)
    .rename_axis("tipopozo")
    .reset_index(name="cantidad_gaps_mayores_1")
)

gaps_por_tipopozo.head(15)

,tipopozo,cantidad_gaps_mayores_1
0,Otro tipo,391
1,Gasífero,230
2,Petrolífero,202
3,NaN,7
4,Sumidero,1


In [84]:
gaps_por_empresa = (
    gaps_mayores_1["empresa"]
    .value_counts(dropna=False)
    .rename_axis("empresa")
    .reset_index(name="cantidad_gaps_mayores_1")
)

gaps_por_empresa.head(15)

,empresa,cantidad_gaps_mayores_1
0,YPF S.A.,345
1,PLUSPETROL S.A.,163
2,SHELL ARGENTINA S.A.,161
3,TOTAL AUSTRAL S.A.,39
4,QUINTANA E&P ARGENTINA S.R.L.,28
5,O&G DEVELOPMENTS LTD S.A.,28
6,TECPETROL S.A.,17
7,APACHE ENERGIA ARGENTINA S.R.L.,11
8,PETROLERA EL TREBOL S.A.,11
9,GAS Y PETROLEO DEL NEUQUEN S.A.,7


### Conclusion sobre continuidad mensual

La gran mayoria de los pares consecutivos por pozo tiene continuidad mensual: 400.236 pares presentan gap de 1 mes, equivalentes al 99,79% de los pares con siguiente registro observado. Solo 831 pares tienen gaps mayores a 1 mes, equivalentes al 0,21%.

Para una primera version exploratoria del proyecto, usar el siguiente registro observado como proxy del mes siguiente es casi equivalente a usar el mes calendario siguiente, porque los gaps son muy poco frecuentes. Sin embargo, desde el punto de vista metodologico, `caida_critica` se define como una caida en el mes siguiente. Por lo tanto, conviene exigir `gap_meses_hasta_siguiente == 1` antes de cerrar la base de modelado.

Decision recomendada para el proyecto del curso: documentar esta validacion y dejar como decision pendiente ajustar el target para que solo se calcule cuando el siguiente registro observado corresponda exactamente al mes calendario siguiente. El impacto esperado en cantidad de datos es bajo, pero mejora la consistencia temporal del target y reduce riesgo de interpretar saltos de varios meses como caidas mensuales.

## Target final con continuidad mensual estricta

A partir de la validacion anterior, se ajusta la construccion formal del target para exigir continuidad mensual estricta. Esto significa que `caida_critica` solo se calcula cuando el siguiente registro observado del mismo pozo corresponde exactamente al mes calendario siguiente (`gap_meses_hasta_siguiente == 1`).

Este ajuste mejora la consistencia temporal del proyecto: evita interpretar saltos de varios meses como si fueran una caida mensual y reduce el riesgo de leakage temporal o de una definicion ambigua del evento objetivo.

### Recalculo estricto del target

Se conserva `df_target` como version previa para comparar. Luego se recalcula el target sobre una copia de `df_fe`, ordenando por `idpozo` y `fecha_data`, y exigiendo que el siguiente registro observado este a un mes calendario de distancia.

In [85]:
df_target_previo = df_target.copy() if "df_target" in globals() else None

df_fe_strict = df_fe.copy()
df_fe_strict["fecha_data"] = pd.to_datetime(df_fe_strict["fecha_data"], errors="coerce")
df_fe_strict = df_fe_strict.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)

df_fe_strict["fecha_siguiente_observada"] = df_fe_strict.groupby("idpozo")["fecha_data"].shift(-1)

mes_actual_idx = df_fe_strict["fecha_data"].dt.year * 12 + df_fe_strict["fecha_data"].dt.month
mes_siguiente_idx = (
    df_fe_strict["fecha_siguiente_observada"].dt.year * 12
    + df_fe_strict["fecha_siguiente_observada"].dt.month
)

df_fe_strict["gap_meses_hasta_siguiente"] = mes_siguiente_idx - mes_actual_idx
df_fe_strict["produccion_principal_next"] = df_fe_strict.groupby("idpozo")["produccion_principal"].shift(-1)
df_fe_strict["var_futura_principal_pct"] = np.nan

strict_target_mask = (
    (df_fe_strict["produccion_principal"] > 0)
    & df_fe_strict["produccion_principal_next"].notna()
    & (df_fe_strict["gap_meses_hasta_siguiente"] == 1)
)

df_fe_strict.loc[strict_target_mask, "var_futura_principal_pct"] = (
    (
        df_fe_strict.loc[strict_target_mask, "produccion_principal_next"]
        - df_fe_strict.loc[strict_target_mask, "produccion_principal"]
    )
    / df_fe_strict.loc[strict_target_mask, "produccion_principal"]
    * 100
)

df_fe_strict["caida_critica"] = pd.Series(pd.NA, index=df_fe_strict.index, dtype="Int64")
df_fe_strict.loc[strict_target_mask, "caida_critica"] = (
    df_fe_strict.loc[strict_target_mask, "var_futura_principal_pct"] <= -TARGET_THRESHOLD_PCT
).astype(int)

df_fe_strict.loc[strict_target_mask, [
    "idpozo", "fecha_data", "fecha_siguiente_observada", "gap_meses_hasta_siguiente",
    "produccion_principal", "produccion_principal_next", "var_futura_principal_pct", "caida_critica"
]].head(10)

,idpozo,fecha_data,fecha_siguiente_observada,gap_meses_hasta_siguiente,produccion_principal,produccion_principal_next,var_futura_principal_pct,caida_critica
1,3640,2006-03-31,2006-04-30,1.00,30.81,76.61,148.67,0
2,3640,2006-04-30,2006-05-31,1.00,76.61,63.27,-17.41,0
3,3640,2006-05-31,2006-06-30,1.00,63.27,55.04,-13.01,0
4,3640,2006-06-30,2006-07-31,1.00,55.04,58.86,6.94,0
5,3640,2006-07-31,2006-08-31,1.00,58.86,33.24,-43.53,1
6,3640,2006-08-31,2006-09-30,1.00,33.24,37.29,12.17,0
7,3640,2006-09-30,2006-10-31,1.00,37.29,64.17,72.08,0
8,3640,2006-10-31,2006-11-30,1.00,64.17,62.86,-2.04,0
9,3640,2006-11-30,2006-12-31,1.00,62.86,66.64,6.02,0
10,3640,2006-12-31,2007-01-31,1.00,66.64,51.73,-22.38,0


### Base final modelable

Se crea `df_target_final` con registros donde el target estricto pudo calcularse. Las columnas que miran el futuro se excluyen del dataset final para que no puedan usarse como features por error.

In [86]:
future_columns_strict = [
    "produccion_principal_next",
    "fecha_siguiente_observada",
    "gap_meses_hasta_siguiente",
    "var_futura_principal_pct",
]

df_target_final = df_fe_strict.loc[strict_target_mask].copy()
df_target_final["caida_critica"] = df_target_final["caida_critica"].astype(int)
df_target_final = df_target_final.drop(columns=[col for col in future_columns_strict if col in df_target_final.columns])

print(f"Dimensiones df_target_final: {df_target_final.shape[0]:,} filas x {df_target_final.shape[1]:,} columnas")

Dimensiones df_target_final: 319,453 filas x 43 columnas


### Comparacion antes y despues del ajuste

Se compara la version previa del target, basada en el siguiente registro observado, contra la version final estricta que exige continuidad mensual exacta.

In [87]:
filas_target_previo = len(df_target_previo) if df_target_previo is not None else np.nan
filas_target_final = len(df_target_final)
registros_eliminados_por_continuidad = filas_target_previo - filas_target_final

resumen_comparativo_target = pd.DataFrame(
    {
        "metrica": [
            "filas_df_target_previo",
            "filas_df_target_final",
            "registros_eliminados_por_exigir_continuidad_mensual",
            "pct_eliminado_sobre_target_previo",
            "fecha_min_df_target_final",
            "fecha_max_df_target_final",
            "pozos_unicos_df_target_final",
        ],
        "valor": [
            filas_target_previo,
            filas_target_final,
            registros_eliminados_por_continuidad,
            registros_eliminados_por_continuidad / filas_target_previo * 100,
            df_target_final["fecha_data"].min(),
            df_target_final["fecha_data"].max(),
            df_target_final["idpozo"].nunique(),
        ],
    }
)

resumen_comparativo_target

,metrica,valor
0,filas_df_target_previo,319721
1,filas_df_target_final,319453
2,registros_eliminados_por_exigir_continuidad_me...,268
3,pct_eliminado_sobre_target_previo,0.08
4,fecha_min_df_target_final,2006-01-31 00:00:00
5,fecha_max_df_target_final,2026-03-31 00:00:00
6,pozos_unicos_df_target_final,4681


In [88]:
def summarize_target_distribution(df_input: pd.DataFrame, version: str) -> pd.DataFrame:
    counts = df_input["caida_critica"].value_counts().sort_index()
    pct = df_input["caida_critica"].value_counts(normalize=True).sort_index() * 100
    return pd.DataFrame(
        {
            "version": version,
            "caida_critica": counts.index.astype(int),
            "cantidad": counts.values,
            "porcentaje": pct.values,
        }
    )


target_distribution_comparison = pd.concat(
    [
        summarize_target_distribution(df_target_previo, "previo_shift_sin_gap_estricto"),
        summarize_target_distribution(df_target_final, "final_gap_1_mes"),
    ],
    ignore_index=True,
)

target_distribution_comparison

,version,caida_critica,cantidad,porcentaje
0,previo_shift_sin_gap_estricto,0,281911,88.17
1,previo_shift_sin_gap_estricto,1,37810,11.83
2,final_gap_1_mes,0,281723,88.19
3,final_gap_1_mes,1,37730,11.81


### Distribucion final de `caida_critica`

Resumen absoluto y porcentual de la clase objetivo final con continuidad mensual estricta.

In [89]:
target_final_distribution = pd.DataFrame(
    {
        "cantidad": df_target_final["caida_critica"].value_counts().sort_index(),
        "porcentaje": df_target_final["caida_critica"].value_counts(normalize=True).sort_index() * 100,
    }
)
target_final_distribution.index.name = "caida_critica"

porcentaje_clase_positiva_final = float(
    target_final_distribution.loc[1, "porcentaje"] if 1 in target_final_distribution.index else 0
)

target_final_distribution

,cantidad,porcentaje
caida_critica,,
0,281723,88.19
1,37730,11.81


In [90]:
target_final_metrics = pd.DataFrame(
    {
        "metrica": [
            "filas_df_target_final",
            "columnas_df_target_final",
            "fecha_min_df_target_final",
            "fecha_max_df_target_final",
            "pozos_unicos_df_target_final",
            "porcentaje_clase_positiva_final",
        ],
        "valor": [
            df_target_final.shape[0],
            df_target_final.shape[1],
            df_target_final["fecha_data"].min(),
            df_target_final["fecha_data"].max(),
            df_target_final["idpozo"].nunique(),
            porcentaje_clase_positiva_final,
        ],
    }
)

target_final_metrics

,metrica,valor
0,filas_df_target_final,319453
1,columnas_df_target_final,43
2,fecha_min_df_target_final,2006-01-31 00:00:00
3,fecha_max_df_target_final,2026-03-31 00:00:00
4,pozos_unicos_df_target_final,4681
5,porcentaje_clase_positiva_final,11.81


### Guardado del dataset final

Se sobrescribe `../data/processed/df_target_caida_critica.csv` con la version final modelable que exige continuidad mensual estricta.

In [91]:
OUTPUT_PATH = Path("../data/processed/df_target_caida_critica.csv")
df_target_final.to_csv(OUTPUT_PATH, index=False)

pd.DataFrame(
    {
        "archivo_guardado": [str(OUTPUT_PATH)],
        "filas": [df_target_final.shape[0]],
        "columnas": [df_target_final.shape[1]],
    }
)

,archivo_guardado,filas,columnas
0,..\data\processed\df_target_caida_critica.csv,319453,43


### Conclusion del target final con continuidad mensual estricta

Se exige `gap_meses_hasta_siguiente == 1` porque el objetivo del proyecto es predecir una caida critica en el mes siguiente. Si se compararan registros separados por varios meses, el target podria representar una caida acumulada entre observaciones no consecutivas y no una caida mensual.

El ajuste elimina 268 registros respecto de la version previa del target, un impacto muy bajo frente al tamano total de la base modelable. El balance final queda practicamente estable: la clase positiva representa aproximadamente el 11,81% de los registros y la clase negativa aproximadamente el 88,19%.

`produccion_principal_next`, `fecha_siguiente_observada`, `gap_meses_hasta_siguiente` y `var_futura_principal_pct` no deben usarse como features del modelo porque contienen informacion del futuro o informacion usada directamente para construir el target. Estas variables solo sirven para definir `caida_critica` y validar su consistencia temporal.

Este ajuste mejora la consistencia temporal del proyecto, fortalece la prevencion de data leakage y deja una base final mas alineada con el enunciado del problema: clasificar si un pozo tendra una caida critica de produccion en el mes calendario siguiente.

## Feature engineering historico - Parte 1

En esta seccion se crean las primeras variables explicativas historicas y contemporaneas para preparar el futuro modelado. No se entrenan modelos, no se realiza split train/test y no se usan variables futuras como features.

Las variables creadas usan informacion disponible en el mes observado o informacion historica acumulada hasta ese mes. Por lo tanto, no introducen leakage temporal.

### 1. Carga del dataset con target final

Se carga `df_target_caida_critica.csv`, que contiene el target final con continuidad mensual estricta y sin columnas futuras.

In [92]:
MODEL_DATA_PATH = Path("../data/processed/df_target_caida_critica.csv")

df_model_base = pd.read_csv(MODEL_DATA_PATH, low_memory=False, parse_dates=["fecha_data", "fechaingreso"])
df_model_base["fecha_data"] = pd.to_datetime(df_model_base["fecha_data"], errors="coerce")

df_model = df_model_base.copy()
df_model = df_model.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)

print(f"Dimensiones iniciales df_model: {df_model.shape[0]:,} filas x {df_model.shape[1]:,} columnas")
df_model.head()

Dimensiones iniciales df_model: 319,453 filas x 43 columnas


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,iny_otro,tef,vida_util,tipoextraccion,tipoestado,tipopozo,observaciones,fechaingreso,rectificado,habilitado,idusuario,empresa,sigla,formprod,profundidad,formacion,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data,produccion_principal,criterio_produccion_principal,caida_critica
0,PEL,2006,3,3640,30.81,1.73,30.57,0.00,0.00,0.00,0.00,10.42,"11,006.72",Bombeo Mecánico,Extracción Efectiva,Petrolífero,NaN,2006-11-22 12:25:57.576696,f,t,361,PETROLERA ENTRE LOMAS S.A.,PPC.Nq.EC-4,VMUT,"2,585.00",vaca muerta,ELO,ENTRE LOMAS,ECL,EL CARACOL,NEUQUINA,Neuquén,-68.45,-37.95,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,AVANZADA,SHALE,2006-03-31,30.81,prod_pet_por_pozo_petrolifero,0
1,PEL,2006,4,3640,76.61,3.71,46.48,0.00,0.00,0.00,0.00,30.00,"11,036.72",Bombeo Mecánico,Extracción Efectiva,Petrolífero,NaN,2006-11-22 13:42:25.403312,f,t,361,PETROLERA ENTRE LOMAS S.A.,PPC.Nq.EC-4,VMUT,"2,585.00",vaca muerta,ELO,ENTRE LOMAS,ECL,EL CARACOL,NEUQUINA,Neuquén,-68.45,-37.95,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,AVANZADA,SHALE,2006-04-30,76.61,prod_pet_por_pozo_petrolifero,0
2,PEL,2006,5,3640,63.27,4.20,50.54,0.00,0.00,0.00,0.00,30.92,"11,067.64",Bombeo Mecánico,Extracción Efectiva,Petrolífero,NaN,2006-11-23 16:43:11.653495,f,t,361,PETROLERA ENTRE LOMAS S.A.,PPC.Nq.EC-4,VMUT,"2,585.00",vaca muerta,ELO,ENTRE LOMAS,ECL,EL CARACOL,NEUQUINA,Neuquén,-68.45,-37.95,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,AVANZADA,SHALE,2006-05-31,63.27,prod_pet_por_pozo_petrolifero,0
3,PEL,2006,6,3640,55.04,4.37,52.06,0.00,0.00,0.00,0.00,29.99,"11,097.63",Bombeo Mecánico,Extracción Efectiva,Petrolífero,NaN,2006-11-24 09:36:18.271222,f,t,361,PETROLERA ENTRE LOMAS S.A.,PPC.Nq.EC-4,VMUT,"2,585.00",vaca muerta,ELO,ENTRE LOMAS,ECL,EL CARACOL,NEUQUINA,Neuquén,-68.45,-37.95,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,AVANZADA,SHALE,2006-06-30,55.04,prod_pet_por_pozo_petrolifero,0
4,PEL,2006,7,3640,58.86,3.73,45.72,0.00,0.00,0.00,0.00,30.99,"11,128.62",Bombeo Mecánico,Extracción Efectiva,Petrolífero,NaN,2006-11-24 09:45:21.872154,f,t,361,PETROLERA ENTRE LOMAS S.A.,PPC.Nq.EC-4,VMUT,"2,585.00",vaca muerta,ELO,ENTRE LOMAS,ECL,EL CARACOL,NEUQUINA,Neuquén,-68.45,-37.95,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,AVANZADA,SHALE,2006-07-31,58.86,prod_pet_por_pozo_petrolifero,1


### 2. Variables temporales calendario

Se crean variables derivadas de la fecha del registro: anio, mes, trimestre, semestre, una etiqueta anio-mes y codificacion ciclica del mes. Estas variables no miran el futuro; solo describen el periodo observado.

In [93]:
new_temporal_features = [
    "anio", "mes", "trimestre", "semestre", "anio_mes", "mes_sin", "mes_cos"
]

df_model["anio"] = df_model["fecha_data"].dt.year
df_model["mes"] = df_model["fecha_data"].dt.month
df_model["trimestre"] = df_model["fecha_data"].dt.quarter
df_model["semestre"] = np.where(df_model["mes"] <= 6, 1, 2)
df_model["anio_mes"] = df_model["fecha_data"].dt.to_period("M").astype(str)
df_model["mes_sin"] = np.sin(2 * np.pi * df_model["mes"] / 12)
df_model["mes_cos"] = np.cos(2 * np.pi * df_model["mes"] / 12)

df_model[["fecha_data"] + new_temporal_features].head()

,fecha_data,anio,mes,trimestre,semestre,anio_mes,mes_sin,mes_cos
0,2006-03-31,2006,3,1,1,2006-03,1.00,0.00
1,2006-04-30,2006,4,2,1,2006-04,0.87,-0.50
2,2006-05-31,2006,5,2,1,2006-05,0.50,-0.87
3,2006-06-30,2006,6,2,1,2006-06,0.00,-1.00
4,2006-07-31,2006,7,3,2,2006-07,-0.50,-0.87


### 3. Antiguedad del pozo

Se calcula la primera fecha disponible de cada pozo en la base modelable y la antiguedad en meses de cada registro respecto de esa fecha. Es una variable historica: para cada fila solo depende del inicio observado del pozo y de la fecha actual del registro.

In [94]:
df_model["fecha_inicio_pozo"] = df_model.groupby("idpozo")["fecha_data"].transform("min")

fecha_actual_idx = df_model["fecha_data"].dt.year * 12 + df_model["fecha_data"].dt.month
fecha_inicio_idx = df_model["fecha_inicio_pozo"].dt.year * 12 + df_model["fecha_inicio_pozo"].dt.month
df_model["antiguedad_meses"] = fecha_actual_idx - fecha_inicio_idx

new_age_features = ["fecha_inicio_pozo", "antiguedad_meses"]

df_model[["idpozo", "fecha_data", "fecha_inicio_pozo", "antiguedad_meses"]].head(10)

,idpozo,fecha_data,fecha_inicio_pozo,antiguedad_meses
0,3640,2006-03-31,2006-03-31,0
1,3640,2006-04-30,2006-03-31,1
2,3640,2006-05-31,2006-03-31,2
3,3640,2006-06-30,2006-03-31,3
4,3640,2006-07-31,2006-03-31,4
5,3640,2006-08-31,2006-03-31,5
6,3640,2006-09-30,2006-03-31,6
7,3640,2006-10-31,2006-03-31,7
8,3640,2006-11-30,2006-03-31,8
9,3640,2006-12-31,2006-03-31,9


### 4. Ratios productivos contemporaneos

Se crean ratios entre producciones del mismo mes observado. Para evitar divisiones por cero, los denominadores menores o iguales a cero se reemplazan por `NaN` antes de calcular el ratio.

In [95]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    safe_denominator = denominator.where(denominator > 0, np.nan)
    return numerator / safe_denominator


df_model["ratio_gas_pet"] = safe_divide(df_model["prod_gas"], df_model["prod_pet"])
df_model["ratio_agua_pet"] = safe_divide(df_model["prod_agua"], df_model["prod_pet"])
df_model["ratio_agua_principal"] = safe_divide(df_model["prod_agua"], df_model["produccion_principal"])

new_ratio_features = ["ratio_gas_pet", "ratio_agua_pet", "ratio_agua_principal"]

df_model[["prod_pet", "prod_gas", "prod_agua", "produccion_principal"] + new_ratio_features].head(10)

,prod_pet,prod_gas,prod_agua,produccion_principal,ratio_gas_pet,ratio_agua_pet,ratio_agua_principal
0,30.81,1.73,30.57,30.81,0.06,0.99,0.99
1,76.61,3.71,46.48,76.61,0.05,0.61,0.61
2,63.27,4.20,50.54,63.27,0.07,0.80,0.80
3,55.04,4.37,52.06,55.04,0.08,0.95,0.95
4,58.86,3.73,45.72,58.86,0.06,0.78,0.78
5,33.24,3.21,62.69,33.24,0.10,1.89,1.89
6,37.29,5.67,53.21,37.29,0.15,1.43,1.43
7,64.17,5.01,38.69,64.17,0.08,0.60,0.60
8,62.86,2.75,52.30,62.86,0.04,0.83,0.83
9,66.64,3.62,65.04,66.64,0.05,0.98,0.98


### 5. Resumen de features creadas

Se revisan dimensiones, columnas nuevas y nulos generados por las variables creadas. Los nulos en ratios son esperables cuando el denominador es cero o no positivo.

In [96]:
new_features_part_1 = new_temporal_features + new_age_features + new_ratio_features

feature_engineering_part_1_summary = pd.DataFrame(
    {
        "metrica": [
            "filas_df_model",
            "columnas_iniciales",
            "columnas_finales",
            "features_nuevas",
        ],
        "valor": [
            df_model.shape[0],
            df_model_base.shape[1],
            df_model.shape[1],
            len(new_features_part_1),
        ],
    }
)

feature_engineering_part_1_summary

,metrica,valor
0,filas_df_model,319453
1,columnas_iniciales,43
2,columnas_finales,53
3,features_nuevas,12


In [97]:
new_features_nulls = pd.DataFrame(
    {
        "feature": new_features_part_1,
        "nulos": [df_model[col].isna().sum() for col in new_features_part_1],
        "pct_nulos": [df_model[col].isna().mean() * 100 for col in new_features_part_1],
        "valores_unicos": [df_model[col].nunique(dropna=True) for col in new_features_part_1],
    }
).sort_values(["pct_nulos", "feature"], ascending=[False, True])

new_features_nulls

,feature,nulos,pct_nulos,valores_unicos
10,ratio_agua_pet,61430,19.23,222040
9,ratio_gas_pet,61430,19.23,253558
0,anio,0,0.00,21
4,anio_mes,0,0.00,243
8,antiguedad_meses,0,0.00,243
7,fecha_inicio_pozo,0,0.00,233
1,mes,0,0.00,12
6,mes_cos,0,0.00,11
5,mes_sin,0,0.00,11
11,ratio_agua_principal,0,0.00,274383


In [98]:
df_model[new_features_part_1 + ["caida_critica"]].head()

,anio,mes,trimestre,semestre,anio_mes,mes_sin,mes_cos,fecha_inicio_pozo,antiguedad_meses,ratio_gas_pet,ratio_agua_pet,ratio_agua_principal,caida_critica
0,2006,3,1,1,2006-03,1.00,0.00,2006-03-31,0,0.06,0.99,0.99,0
1,2006,4,2,1,2006-04,0.87,-0.50,2006-03-31,1,0.05,0.61,0.61,0
2,2006,5,2,1,2006-05,0.50,-0.87,2006-03-31,2,0.07,0.80,0.80,0
3,2006,6,2,1,2006-06,0.00,-1.00,2006-03-31,3,0.08,0.95,0.95,0
4,2006,7,3,2,2006-07,-0.50,-0.87,2006-03-31,4,0.06,0.78,0.78,1


### Conclusion de feature engineering historico - Parte 1

Se crearon variables temporales (`anio`, `mes`, `trimestre`, `semestre`, `anio_mes`, `mes_sin`, `mes_cos`), variables de antiguedad del pozo (`fecha_inicio_pozo`, `antiguedad_meses`) y ratios productivos contemporaneos (`ratio_gas_pet`, `ratio_agua_pet`, `ratio_agua_principal`).

Estas variables no generan leakage porque no usan informacion del mes siguiente ni variables utilizadas para construir el target. Las variables calendario describen el mes actual, la antiguedad usa el historial disponible hasta el registro observado y los ratios se calculan con producciones del mismo mes.

Queda pendiente para la siguiente parte crear features historicas por pozo, como lags, medias moviles, tendencias y variaciones pasadas, cuidando que todas usen exclusivamente informacion anterior o disponible hasta el mes observado.

## Feature engineering historico - Parte 2

En esta seccion se agregan features historicas por pozo: lags, variaciones historicas, medias moviles, acumulados y flags simples.

Todas las variables se calculan agrupando por `idpozo` y ordenando por `fecha_data`. Las features usan el mes actual y/o meses anteriores; nunca usan el mes siguiente ni variables futuras utilizadas para construir el target.

### 1. Orden temporal por pozo

Antes de calcular features historicas se vuelve a ordenar `df_model` por `idpozo` y `fecha_data`. Esto asegura que los lags, acumulados y ventanas moviles respeten la secuencia temporal de cada pozo.

In [99]:
df_model = df_model.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)
pozo_group = df_model.groupby("idpozo", group_keys=False)
columnas_antes_parte_2 = df_model.shape[1]

print(f"Dimensiones antes de Parte 2: {df_model.shape[0]:,} filas x {df_model.shape[1]:,} columnas")

Dimensiones antes de Parte 2: 319,453 filas x 53 columnas


### 2. Lags de produccion

Los lags representan valores observados en meses anteriores del mismo pozo. Son features historicas y no generan leakage porque no usan informacion futura.

In [100]:
lag_specs = {
    "produccion_principal": [1, 2, 3, 6],
    "prod_pet": [1, 3],
    "prod_gas": [1, 3],
    "prod_agua": [1, 3],
}

new_lag_features = []

for base_col, lags in lag_specs.items():
    for lag in lags:
        feature_name = f"{base_col}_lag_{lag}"
        df_model[feature_name] = pozo_group[base_col].shift(lag)
        new_lag_features.append(feature_name)

df_model[["idpozo", "fecha_data", "produccion_principal"] + new_lag_features[:4]].head(12)

,idpozo,fecha_data,produccion_principal,produccion_principal_lag_1,produccion_principal_lag_2,produccion_principal_lag_3,produccion_principal_lag_6
0,3640,2006-03-31,30.81,NaN,NaN,NaN,NaN
1,3640,2006-04-30,76.61,30.81,NaN,NaN,NaN
2,3640,2006-05-31,63.27,76.61,30.81,NaN,NaN
3,3640,2006-06-30,55.04,63.27,76.61,30.81,NaN
4,3640,2006-07-31,58.86,55.04,63.27,76.61,NaN
5,3640,2006-08-31,33.24,58.86,55.04,63.27,NaN
6,3640,2006-09-30,37.29,33.24,58.86,55.04,30.81
7,3640,2006-10-31,64.17,37.29,33.24,58.86,76.61
8,3640,2006-11-30,62.86,64.17,37.29,33.24,63.27
9,3640,2006-12-31,66.64,62.86,64.17,37.29,55.04


### 3. Variaciones historicas porcentuales

Se calculan variaciones porcentuales contra valores pasados. Para evitar divisiones por cero, la variacion solo se calcula cuando el valor historico usado como denominador es mayor a cero.

In [101]:
def pct_variation_from_lag(current: pd.Series, lagged: pd.Series) -> pd.Series:
    denominator = lagged.where(lagged > 0, np.nan)
    return (current - lagged) / denominator * 100


df_model["var_principal_1m_pct"] = pct_variation_from_lag(
    df_model["produccion_principal"], df_model["produccion_principal_lag_1"]
)
df_model["var_principal_3m_pct"] = pct_variation_from_lag(
    df_model["produccion_principal"], df_model["produccion_principal_lag_3"]
)
df_model["var_pet_1m_pct"] = pct_variation_from_lag(df_model["prod_pet"], df_model["prod_pet_lag_1"])
df_model["var_gas_1m_pct"] = pct_variation_from_lag(df_model["prod_gas"], df_model["prod_gas_lag_1"])
df_model["var_agua_1m_pct"] = pct_variation_from_lag(df_model["prod_agua"], df_model["prod_agua_lag_1"])

new_variation_features = [
    "var_principal_1m_pct",
    "var_principal_3m_pct",
    "var_pet_1m_pct",
    "var_gas_1m_pct",
    "var_agua_1m_pct",
]

df_model[["idpozo", "fecha_data", "produccion_principal"] + new_variation_features].head(12)

,idpozo,fecha_data,produccion_principal,var_principal_1m_pct,var_principal_3m_pct,var_pet_1m_pct,var_gas_1m_pct,var_agua_1m_pct
0,3640,2006-03-31,30.81,NaN,NaN,NaN,NaN,NaN
1,3640,2006-04-30,76.61,148.67,NaN,148.67,114.50,52.04
2,3640,2006-05-31,63.27,-17.41,NaN,-17.41,13.17,8.73
3,3640,2006-06-30,55.04,-13.01,78.66,-13.01,4.07,3.02
4,3640,2006-07-31,58.86,6.94,-23.17,6.94,-14.66,-12.19
5,3640,2006-08-31,33.24,-43.53,-47.46,-43.53,-13.93,37.11
6,3640,2006-09-30,37.29,12.17,-32.26,12.17,76.49,-15.12
7,3640,2006-10-31,64.17,72.08,9.01,72.08,-11.70,-27.28
8,3640,2006-11-30,62.86,-2.04,89.09,-2.04,-44.97,35.18
9,3640,2006-12-31,66.64,6.02,78.73,6.02,31.36,24.35


### 4. Medias moviles y volatilidad historica

Se calculan medias moviles y desvios estandar usando ventanas que incluyen el mes actual y meses anteriores del mismo pozo. Estas features resumen el comportamiento reciente disponible al momento de observacion.

In [102]:
df_model["produccion_principal_mm_3"] = pozo_group["produccion_principal"].transform(
    lambda s: s.rolling(window=3, min_periods=1).mean()
)
df_model["produccion_principal_mm_6"] = pozo_group["produccion_principal"].transform(
    lambda s: s.rolling(window=6, min_periods=1).mean()
)
df_model["produccion_principal_std_3"] = pozo_group["produccion_principal"].transform(
    lambda s: s.rolling(window=3, min_periods=2).std()
)
df_model["produccion_principal_std_6"] = pozo_group["produccion_principal"].transform(
    lambda s: s.rolling(window=6, min_periods=2).std()
)
df_model["prod_pet_mm_3"] = pozo_group["prod_pet"].transform(lambda s: s.rolling(window=3, min_periods=1).mean())
df_model["prod_gas_mm_3"] = pozo_group["prod_gas"].transform(lambda s: s.rolling(window=3, min_periods=1).mean())
df_model["prod_agua_mm_3"] = pozo_group["prod_agua"].transform(lambda s: s.rolling(window=3, min_periods=1).mean())

new_rolling_features = [
    "produccion_principal_mm_3",
    "produccion_principal_mm_6",
    "produccion_principal_std_3",
    "produccion_principal_std_6",
    "prod_pet_mm_3",
    "prod_gas_mm_3",
    "prod_agua_mm_3",
]

df_model[["idpozo", "fecha_data", "produccion_principal"] + new_rolling_features].head(12)

,idpozo,fecha_data,produccion_principal,produccion_principal_mm_3,produccion_principal_mm_6,produccion_principal_std_3,produccion_principal_std_6,prod_pet_mm_3,prod_gas_mm_3,prod_agua_mm_3
0,3640,2006-03-31,30.81,30.81,30.81,NaN,NaN,30.81,1.73,30.57
1,3640,2006-04-30,76.61,53.71,53.71,32.39,32.39,53.71,2.72,38.52
2,3640,2006-05-31,63.27,56.90,56.90,23.56,23.56,56.90,3.22,42.53
3,3640,2006-06-30,55.04,64.98,56.43,10.88,19.26,64.98,4.10,49.69
4,3640,2006-07-31,58.86,59.06,56.92,4.12,16.71,59.06,4.10,49.44
5,3640,2006-08-31,33.24,49.05,52.97,13.82,17.80,49.05,3.77,53.49
6,3640,2006-09-30,37.29,43.13,54.05,13.77,16.32,43.13,4.20,53.87
7,3640,2006-10-31,64.17,44.90,51.98,16.81,13.41,44.90,4.63,51.53
8,3640,2006-11-30,62.86,54.77,51.91,15.15,13.35,54.77,4.48,48.07
9,3640,2006-12-31,66.64,64.56,53.84,1.92,14.67,64.56,3.79,52.01


### 5. Acumulados historicos

Los acumulados se calculan con suma acumulada por pozo hasta el mes observado. Usan el historial disponible hasta la fila actual.

In [103]:
df_model["produccion_principal_acum"] = pozo_group["produccion_principal"].cumsum()
df_model["prod_pet_acum"] = pozo_group["prod_pet"].cumsum()
df_model["prod_gas_acum"] = pozo_group["prod_gas"].cumsum()
df_model["prod_agua_acum"] = pozo_group["prod_agua"].cumsum()

new_accumulated_features = [
    "produccion_principal_acum",
    "prod_pet_acum",
    "prod_gas_acum",
    "prod_agua_acum",
]

df_model[["idpozo", "fecha_data"] + new_accumulated_features].head(12)

,idpozo,fecha_data,produccion_principal_acum,prod_pet_acum,prod_gas_acum,prod_agua_acum
0,3640,2006-03-31,30.81,30.81,1.73,30.57
1,3640,2006-04-30,107.42,107.42,5.44,77.05
2,3640,2006-05-31,170.69,170.69,9.65,127.59
3,3640,2006-06-30,225.73,225.73,14.02,179.65
4,3640,2006-07-31,284.59,284.59,17.75,225.37
5,3640,2006-08-31,317.84,317.84,20.96,288.05
6,3640,2006-09-30,355.12,355.12,26.63,341.26
7,3640,2006-10-31,419.29,419.29,31.64,379.95
8,3640,2006-11-30,482.15,482.15,34.39,432.25
9,3640,2006-12-31,548.79,548.79,38.01,497.28


### 6. Flags operativos y productivos

Se crean indicadores binarios simples para ceros de produccion, tipo de pozo y estado operativo. Estos flags describen informacion disponible en el mes observado.

In [104]:
df_model["flag_prod_principal_cero"] = (df_model["produccion_principal"] == 0).astype(int)
df_model["flag_prod_pet_cero"] = (df_model["prod_pet"] == 0).astype(int)
df_model["flag_prod_gas_cero"] = (df_model["prod_gas"] == 0).astype(int)
df_model["flag_prod_agua_cero"] = (df_model["prod_agua"] == 0).astype(int)
df_model["flag_pozo_gasifero"] = df_model["criterio_produccion_principal"].eq("prod_gas_por_pozo_gasifero").astype(int)
df_model["flag_pozo_petrolifero"] = df_model["criterio_produccion_principal"].eq("prod_pet_por_pozo_petrolifero").astype(int)

new_flag_features = [
    "flag_prod_principal_cero",
    "flag_prod_pet_cero",
    "flag_prod_gas_cero",
    "flag_prod_agua_cero",
    "flag_pozo_gasifero",
    "flag_pozo_petrolifero",
]

if "tipoestado" in df_model.columns:
    df_model["flag_extraccion_efectiva"] = df_model["tipoestado"].map(normalize_text_for_matching).eq("extraccion efectiva").astype(int)
    new_flag_features.append("flag_extraccion_efectiva")

df_model[new_flag_features].head()

,flag_prod_principal_cero,flag_prod_pet_cero,flag_prod_gas_cero,flag_prod_agua_cero,flag_pozo_gasifero,flag_pozo_petrolifero,flag_extraccion_efectiva
0,0,0,0,0,0,1,1
1,0,0,0,0,0,1,1
2,0,0,0,0,0,1,1
3,0,0,0,0,0,1,1
4,0,0,0,0,0,1,1


### 7. Nulos generados por features historicas

Los lags y variaciones generan nulos al inicio del historial de cada pozo. Los desvios estandar tambien generan nulos cuando todavia no hay suficientes observaciones en la ventana. Estos nulos se imputaran mas adelante dentro del pipeline de modelado.

In [105]:
new_features_part_2 = (
    new_lag_features
    + new_variation_features
    + new_rolling_features
    + new_accumulated_features
    + new_flag_features
)

feature_engineering_part_2_summary = pd.DataFrame(
    {
        "metrica": [
            "filas_df_model",
            "columnas_despues_parte_1",
            "columnas_despues_parte_2",
            "features_nuevas_parte_2",
        ],
        "valor": [
            df_model.shape[0],
            columnas_antes_parte_2,
            df_model.shape[1],
            len(new_features_part_2),
        ],
    }
)

feature_engineering_part_2_summary

,metrica,valor
0,filas_df_model,319453
1,columnas_despues_parte_1,53
2,columnas_despues_parte_2,86
3,features_nuevas_parte_2,33


In [106]:
new_features_part_2_nulls = pd.DataFrame(
    {
        "feature": new_features_part_2,
        "nulos": [df_model[col].isna().sum() for col in new_features_part_2],
        "pct_nulos": [df_model[col].isna().mean() * 100 for col in new_features_part_2],
        "valores_unicos": [df_model[col].nunique(dropna=True) for col in new_features_part_2],
    }
).sort_values(["pct_nulos", "feature"], ascending=[False, True])

new_features_part_2_nulls

,feature,nulos,pct_nulos,valores_unicos
12,var_pet_1m_pct,65374,20.46,236144
14,var_agua_1m_pct,44504,13.93,247577
3,produccion_principal_lag_6,27175,8.51,192756
9,prod_agua_lag_3,13830,4.33,120239
7,prod_gas_lag_3,13830,4.33,176497
5,prod_pet_lag_3,13830,4.33,129876
2,produccion_principal_lag_3,13830,4.33,200872
11,var_principal_3m_pct,13830,4.33,300941
1,produccion_principal_lag_2,9292,2.91,203628
13,var_gas_1m_pct,6676,2.09,306964


In [107]:
df_model[["idpozo", "fecha_data", "produccion_principal", "caida_critica"] + new_features_part_2[:12]].head(15)

,idpozo,fecha_data,produccion_principal,caida_critica,produccion_principal_lag_1,produccion_principal_lag_2,produccion_principal_lag_3,produccion_principal_lag_6,prod_pet_lag_1,prod_pet_lag_3,prod_gas_lag_1,prod_gas_lag_3,prod_agua_lag_1,prod_agua_lag_3,var_principal_1m_pct,var_principal_3m_pct
0,3640,2006-03-31,30.81,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3640,2006-04-30,76.61,0,30.81,NaN,NaN,NaN,30.81,NaN,1.73,NaN,30.57,NaN,148.67,NaN
2,3640,2006-05-31,63.27,0,76.61,30.81,NaN,NaN,76.61,NaN,3.71,NaN,46.48,NaN,-17.41,NaN
3,3640,2006-06-30,55.04,0,63.27,76.61,30.81,NaN,63.27,30.81,4.20,1.73,50.54,30.57,-13.01,78.66
4,3640,2006-07-31,58.86,1,55.04,63.27,76.61,NaN,55.04,76.61,4.37,3.71,52.06,46.48,6.94,-23.17
5,3640,2006-08-31,33.24,0,58.86,55.04,63.27,NaN,58.86,63.27,3.73,4.20,45.72,50.54,-43.53,-47.46
6,3640,2006-09-30,37.29,0,33.24,58.86,55.04,30.81,33.24,55.04,3.21,4.37,62.69,52.06,12.17,-32.26
7,3640,2006-10-31,64.17,0,37.29,33.24,58.86,76.61,37.29,58.86,5.67,3.73,53.21,45.72,72.08,9.01
8,3640,2006-11-30,62.86,0,64.17,37.29,33.24,63.27,64.17,33.24,5.01,3.21,38.69,62.69,-2.04,89.09
9,3640,2006-12-31,66.64,0,62.86,64.17,37.29,55.04,62.86,37.29,2.75,5.67,52.30,53.21,6.02,78.73


### Conclusion de feature engineering historico - Parte 2

Se crearon features historicas por pozo: lags de produccion, variaciones porcentuales contra meses previos, medias moviles, desvios estandar, acumulados y flags productivos/operativos.

Estas features respetan la temporalidad porque se calculan agrupando por `idpozo` y ordenando por `fecha_data`. Los lags usan meses anteriores, las variaciones comparan contra valores pasados, las ventanas moviles resumen informacion disponible hasta el mes observado y los acumulados suman produccion hasta la fila actual. No se usa informacion del mes siguiente.

Quedan nulos esperables en lags, variaciones y desvios estandar, especialmente al inicio de la historia de cada pozo o cuando el denominador historico es cero. Esos nulos no se imputan en este notebook: deberan tratarse luego dentro del pipeline de modelado para evitar leakage y mantener reproducibilidad.

## Dataset final de modelado

En esta seccion se arma el dataset final para modelado a partir de `df_model`. No se entrenan modelos, no se hace split train/test y no se imputan nulos. La imputacion y el preprocesamiento final se haran luego dentro del pipeline de modelado.

El objetivo de este bloque es dejar una tabla clara, trazable y sin variables futuras como features.

### 1. Definicion de target, columnas de trazabilidad y columnas excluidas

`idpozo` y `fecha_data` se conservan solo para trazabilidad y control temporal. No deben usarse como variables predictoras directas en el modelo. Las columnas futuras y auxiliares se excluyen para prevenir data leakage.

In [108]:
target_col = "caida_critica"

leakage_cols = [
    "produccion_principal_next",
    "fecha_siguiente_observada",
    "gap_meses_hasta_siguiente",
    "var_futura_principal_pct",
    "prod_pet_next",
    "prod_gas_next",
]

id_cols = ["idpozo", "fecha_data"]

high_missing_or_free_text_cols = [
    "vida_util",
    "observaciones",
    "sigla",
]

auxiliary_or_administrative_cols = [
    "fechaingreso",
    "fecha_inicio_pozo",
    "criterio_produccion_principal",
    "idusuario",
    "rectificado",
    "habilitado",
]

excluded_cols = sorted(
    set(leakage_cols + high_missing_or_free_text_cols + auxiliary_or_administrative_cols)
)

excluded_cols_present = [col for col in excluded_cols if col in df_model.columns]
leakage_cols_present = [col for col in leakage_cols if col in df_model.columns]

pd.DataFrame(
    {
        "columna_excluida": excluded_cols_present,
        "motivo": [
            "leakage/futura" if col in leakage_cols else "auxiliar/administrativa/texto_libre/alta_nulidad"
            for col in excluded_cols_present
        ],
    }
)

,columna_excluida,motivo
0,criterio_produccion_principal,auxiliar/administrativa/texto_libre/alta_nulidad
1,fecha_inicio_pozo,auxiliar/administrativa/texto_libre/alta_nulidad
2,fechaingreso,auxiliar/administrativa/texto_libre/alta_nulidad
3,habilitado,auxiliar/administrativa/texto_libre/alta_nulidad
4,idusuario,auxiliar/administrativa/texto_libre/alta_nulidad
5,observaciones,auxiliar/administrativa/texto_libre/alta_nulidad
6,rectificado,auxiliar/administrativa/texto_libre/alta_nulidad
7,sigla,auxiliar/administrativa/texto_libre/alta_nulidad
8,vida_util,auxiliar/administrativa/texto_libre/alta_nulidad


### 2. Separacion de features numericas y categoricas

Se excluyen target, columnas de trazabilidad y columnas marcadas como leakage o auxiliares. Luego se separan las features restantes por tipo de dato.

In [109]:
columns_to_exclude_from_features = set(id_cols + [target_col] + excluded_cols_present)

feature_candidates = [
    col for col in df_model.columns
    if col not in columns_to_exclude_from_features
]

numeric_features = df_model[feature_candidates].select_dtypes(include=np.number).columns.tolist()
categorical_features = df_model[feature_candidates].select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

selected_feature_cols = numeric_features + categorical_features
non_selected_feature_candidates = [
    col for col in feature_candidates
    if col not in selected_feature_cols
]

feature_type_summary = pd.DataFrame(
    {
        "tipo": [
            "features_numericas",
            "features_categoricas",
            "candidatas_no_seleccionadas_por_tipo",
        ],
        "cantidad": [
            len(numeric_features),
            len(categorical_features),
            len(non_selected_feature_candidates),
        ],
    }
)

feature_type_summary

,tipo,cantidad
0,features_numericas,55
1,features_categoricas,19
2,candidatas_no_seleccionadas_por_tipo,0


In [110]:
pd.DataFrame({"numeric_features": numeric_features})

,numeric_features
0,anio
1,mes
2,prod_pet
3,prod_gas
4,prod_agua
5,iny_agua
6,iny_gas
7,iny_co2
8,iny_otro
9,tef


In [111]:
pd.DataFrame({"categorical_features": categorical_features})

,categorical_features
0,idempresa
1,tipoextraccion
2,tipoestado
3,tipopozo
4,empresa
5,formprod
6,formacion
7,idareapermisoconcesion
8,areapermisoconcesion
9,idareayacimiento


### 3. Creacion de `df_model_ready`

La tabla final conserva columnas de trazabilidad, features numericas, features categoricas y el target. Los nulos se mantienen sin imputar para tratarlos luego dentro del pipeline.

In [112]:
df_model_ready = df_model[id_cols + numeric_features + categorical_features + [target_col]].copy()

model_ready_summary = pd.DataFrame(
    {
        "metrica": [
            "filas_df_model_ready",
            "columnas_df_model_ready",
            "features_numericas",
            "features_categoricas",
            "columnas_trazabilidad",
        ],
        "valor": [
            df_model_ready.shape[0],
            df_model_ready.shape[1],
            len(numeric_features),
            len(categorical_features),
            len(id_cols),
        ],
    }
)

model_ready_summary

,metrica,valor
0,filas_df_model_ready,319453
1,columnas_df_model_ready,77
2,features_numericas,55
3,features_categoricas,19
4,columnas_trazabilidad,2


In [113]:
model_target_distribution = pd.DataFrame(
    {
        "cantidad": df_model_ready[target_col].value_counts().sort_index(),
        "porcentaje": df_model_ready[target_col].value_counts(normalize=True).sort_index() * 100,
    }
)
model_target_distribution.index.name = target_col

model_target_distribution

,cantidad,porcentaje
caida_critica,,
0,281723,88.19
1,37730,11.81


In [114]:
model_ready_nulls = pd.DataFrame(
    {
        "columna": selected_feature_cols,
        "nulos": [df_model_ready[col].isna().sum() for col in selected_feature_cols],
        "pct_nulos": [df_model_ready[col].isna().mean() * 100 for col in selected_feature_cols],
        "tipo_dato": [str(df_model_ready[col].dtype) for col in selected_feature_cols],
    }
).query("nulos > 0").sort_values(["pct_nulos", "nulos"], ascending=False)

model_ready_nulls.head(30)

,columna,nulos,pct_nulos,tipo_dato
34,var_pet_1m_pct,65374,20.46,float64
19,ratio_gas_pet,61430,19.23,float64
20,ratio_agua_pet,61430,19.23,float64
36,var_agua_1m_pct,44504,13.93,float64
25,produccion_principal_lag_6,27175,8.51,float64
24,produccion_principal_lag_3,13830,4.33,float64
27,prod_pet_lag_3,13830,4.33,float64
29,prod_gas_lag_3,13830,4.33,float64
31,prod_agua_lag_3,13830,4.33,float64
33,var_principal_3m_pct,13830,4.33,float64


In [115]:
pd.DataFrame({"columna_excluida_por_leakage": leakage_cols_present})

,columna_excluida_por_leakage


### 4. Guardado de `model_dataset.csv`

Se guarda el dataset final de modelado. Este archivo todavia contiene nulos; la imputacion se realizara despues dentro del pipeline para evitar leakage y mantener reproducibilidad.

In [116]:
MODEL_OUTPUT_PATH = Path("../data/processed/model_dataset.csv")
df_model_ready.to_csv(MODEL_OUTPUT_PATH, index=False)

pd.DataFrame(
    {
        "archivo_guardado": [str(MODEL_OUTPUT_PATH)],
        "filas": [df_model_ready.shape[0]],
        "columnas": [df_model_ready.shape[1]],
        "features_numericas": [len(numeric_features)],
        "features_categoricas": [len(categorical_features)],
    }
)

,archivo_guardado,filas,columnas,features_numericas,features_categoricas
0,..\data\processed\model_dataset.csv,319453,77,55,19


## Conclusiones de feature engineering

- **Features creadas:** se construyeron variables calendario, antiguedad del pozo, ratios productivos, lags, variaciones historicas, medias moviles, desvios estandar, acumulados y flags operativos/productivos.
- **Prevencion de leakage:** se excluyeron variables futuras o usadas para construir el target, como `produccion_principal_next`, `fecha_siguiente_observada`, `gap_meses_hasta_siguiente`, `var_futura_principal_pct`, `prod_pet_next` y `prod_gas_next`. Las features historicas se calcularon por `idpozo`, ordenando por `fecha_data`, usando solo el mes actual y meses anteriores.
- **Columnas excluidas:** se excluyeron columnas futuras, columnas auxiliares/administrativas y campos poco adecuados para este primer dataset de modelado, como `vida_util`, `observaciones`, `sigla`, `fechaingreso`, `fecha_inicio_pozo`, `criterio_produccion_principal`, `idusuario`, `rectificado` y `habilitado` cuando estaban presentes.
- **Nulos pendientes:** quedan nulos esperables en lags, variaciones, desvios estandar y ratios con denominadores no positivos. No se imputaron en este notebook; se imputaran posteriormente dentro del pipeline de modelado.
- **Trazabilidad:** `idpozo` y `fecha_data` se conservan en `df_model_ready` para auditoria, seguimiento temporal y validacion de splits, pero no deben usarse como features directas del modelo.
- **Estado del dataset:** `model_dataset.csv` queda listo como base de entrada para el notebook de modelado, pendiente de split temporal, imputacion, encoding, escalado si corresponde y entrenamiento de modelos.